# 13: Word Embeddings - Words Have Meaning

## Beyond One-Hot Encoding

Remember one-hot encoding? It has problems:
- **Huge vectors**: 50,000 dimensions for a typical vocabulary
- **No similarity**: "cat" and "dog" are as different as "cat" and "democracy"
- **Sparse**: Mostly zeros, wastes computation

**Embeddings** solve these problems by representing words as dense vectors where **similar words have similar vectors**.

### The Web Dev Analogy

Embeddings are like **coordinates in a feature space**:
- One-hot = street address (unique but no relation)
- Embedding = GPS coordinates (nearby things have similar coords)

## What You'll Learn
- [ ] Explain why dense embeddings are better than one-hot encoding
- [ ] Compute cosine similarity between word vectors
- [ ] Visualize word relationships in embedding space

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 1**: One-hot encoding | One-hot vectors are sparse and treat all words as equally different — embeddings fix this |
| **Lesson 5**: Bag-of-words features | BoW lost word meaning and similarity — embeddings capture both |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

print("Ready to explore embeddings! 📚")

## 1. The Problem with One-Hot

In [ ]:
# Simple vocabulary
vocab = ['cat', 'dog', 'bird', 'fish', 'car', 'truck', 'bike']
word_to_idx = {word: i for i, word in enumerate(vocab)}

def one_hot(word):
    vec = np.zeros(len(vocab))
    vec[word_to_idx[word]] = 1
    return vec

# One-hot vectors
cat_oh = one_hot('cat')
dog_oh = one_hot('dog')
car_oh = one_hot('car')

print("One-hot vectors:")
print(f"cat: {cat_oh}")
print(f"dog: {dog_oh}")
print(f"car: {car_oh}")

# Cosine similarity
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

print(f"\nSimilarity (cat, dog): {cosine_sim(cat_oh, dog_oh):.3f}")
print(f"Similarity (cat, car): {cosine_sim(cat_oh, car_oh):.3f}")
print("\n❌ Both are 0! Cat-dog should be more similar than cat-car!")

## 2. Embeddings: Dense, Meaningful Vectors

In [ ]:
# Create an embedding layer
vocab_size = len(vocab)
embedding_dim = 4  # Each word is a 4-dimensional vector

embedding = nn.Embedding(vocab_size, embedding_dim)

print(f"Embedding matrix shape: {embedding.weight.shape}")
print(f"  {vocab_size} words × {embedding_dim} dimensions")
print(f"\nEmbedding matrix (random initialization):")
print(embedding.weight.data)

In [ ]:
# Look up embeddings
word_indices = torch.tensor([word_to_idx['cat'], word_to_idx['dog'], word_to_idx['car']])

embedded = embedding(word_indices)

print("Embedded vectors (4 dimensions each):")
for word, vec in zip(['cat', 'dog', 'car'], embedded):
    print(f"{word}: {vec.detach().numpy().round(3)}")

print(f"\n✅ Dense! Only {embedding_dim} values instead of {vocab_size}")

## 3. Embeddings Learn Meaning

The key insight: **embeddings are learned** during training. Words that appear in similar contexts get similar vectors!

In [ ]:
# Let's manually set embeddings to show the concept
# Animals cluster together, vehicles cluster together

learned_embeddings = torch.tensor([
    [1.0, 1.0, 0.0, 0.0],   # cat - animal
    [0.9, 1.1, 0.0, 0.1],   # dog - animal (similar to cat)
    [1.1, 0.9, 0.1, 0.0],   # bird - animal
    [0.8, 1.0, 0.2, 0.0],   # fish - animal
    [0.0, 0.0, 1.0, 1.0],   # car - vehicle
    [0.0, 0.1, 1.1, 0.9],   # truck - vehicle (similar to car)
    [0.1, 0.0, 0.9, 1.1],   # bike - vehicle
])

# Compute similarities
def tensor_cosine_sim(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

print("Similarities with meaningful embeddings:")
print("-" * 40)

pairs = [('cat', 'dog'), ('cat', 'bird'), ('cat', 'car'), ('car', 'truck')]
for w1, w2 in pairs:
    sim = tensor_cosine_sim(
        learned_embeddings[word_to_idx[w1]], 
        learned_embeddings[word_to_idx[w2]]
    )
    print(f"  {w1} - {w2}: {sim:.3f}")

print("\n✅ Now cat-dog is similar, cat-car is different!")

In [ ]:
# Visualize in 2D
from sklearn.decomposition import PCA

# Reduce to 2D for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(learned_embeddings.numpy())

plt.figure(figsize=(10, 8))

# Plot points
colors = ['red', 'red', 'red', 'red', 'blue', 'blue', 'blue']
for i, (word, color) in enumerate(zip(vocab, colors)):
    plt.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], c=color, s=200)
    plt.annotate(word, (embeddings_2d[i, 0] + 0.05, embeddings_2d[i, 1] + 0.05), fontsize=14)

# Add legend
plt.scatter([], [], c='red', label='Animals')
plt.scatter([], [], c='blue', label='Vehicles')

plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.title('Word Embeddings in 2D Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("💡 Similar words cluster together in embedding space!")

## 4. Using Embeddings in Neural Networks

In [ ]:
# Simple text classification with embeddings
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # x: (batch_size, seq_len) - token indices
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        
        # Average over sequence (simple pooling)
        pooled = embedded.mean(dim=1)  # (batch_size, embedding_dim)
        
        # Classification layers
        hidden = self.relu(self.fc1(pooled))
        output = self.fc2(hidden)
        return output

# Create model
model = TextClassifier(
    vocab_size=1000, 
    embedding_dim=64, 
    hidden_dim=32, 
    output_dim=2
)

# Simulate input: batch of 3 sentences, each 5 tokens
sample_input = torch.randint(0, 1000, (3, 5))
print(f"Input shape: {sample_input.shape}")

output = model(sample_input)
print(f"Output shape: {output.shape}")
print(f"\nThis model can classify text into 2 categories!")

## 5. Pre-trained Embeddings

Training embeddings from scratch requires lots of data. Instead, we often use **pre-trained embeddings**:
- **Word2Vec** (Google)
- **GloVe** (Stanford)
- **FastText** (Facebook)

In [ ]:
# Simulate loading pre-trained embeddings
def load_pretrained_embeddings(vocab, embedding_dim=50):
    """
    In practice, you'd load from a file like GloVe.
    Here we simulate meaningful embeddings.
    """
    embeddings = {}
    
    # Simulate semantic groups
    animal_base = np.random.randn(embedding_dim) * 0.1
    vehicle_base = np.random.randn(embedding_dim) * 0.1
    
    for word in vocab:
        if word in ['cat', 'dog', 'bird', 'fish']:
            embeddings[word] = animal_base + np.random.randn(embedding_dim) * 0.05
        else:
            embeddings[word] = vehicle_base + np.random.randn(embedding_dim) * 0.05
    
    return embeddings

pretrained = load_pretrained_embeddings(vocab)
print(f"Loaded embeddings for {len(pretrained)} words")
print(f"Each embedding has {len(pretrained['cat'])} dimensions")

In [ ]:
# Initialize embedding layer with pre-trained weights
def create_embedding_matrix(vocab, pretrained_embeddings, embedding_dim):
    """Create embedding matrix from pre-trained embeddings."""
    matrix = np.zeros((len(vocab), embedding_dim))
    
    for i, word in enumerate(vocab):
        if word in pretrained_embeddings:
            matrix[i] = pretrained_embeddings[word]
        else:
            # Random initialization for unknown words
            matrix[i] = np.random.randn(embedding_dim) * 0.1
    
    return torch.FloatTensor(matrix)

# Create and load
embedding_matrix = create_embedding_matrix(vocab, pretrained, embedding_dim=50)

embedding_layer = nn.Embedding(len(vocab), 50)
embedding_layer.weight.data.copy_(embedding_matrix)

# Optionally freeze (don't update during training)
embedding_layer.weight.requires_grad = False

print("✅ Embedding layer initialized with pre-trained weights!")
print(f"   Frozen: {not embedding_layer.weight.requires_grad}")

## 📝 Check Your Understanding

1. Why are embeddings better than one-hot encoding?
2. What does it mean for similar words to have similar vectors?
3. How are embeddings learned?
4. What are pre-trained embeddings and why use them?
5. When would you freeze vs fine-tune embeddings?

In [ ]:
# --- Exercise 1: Cosine Similarity ---
# Implement cosine similarity and test it.
# cosine_sim(a, b) = dot(a, b) / (||a|| × ||b||)

# YOUR CODE HERE:
def cosine_sim(a, b):
    return None  # Implement the formula

# --- Check ---
assert cosine_sim is not None, "Implement the function!"
# Identical vectors → similarity = 1.0
v1 = np.array([1, 0, 0])
assert abs(cosine_sim(v1, v1) - 1.0) < 0.001, "Identical vectors should have similarity 1.0"
# Orthogonal vectors → similarity = 0.0
v2 = np.array([0, 1, 0])
assert abs(cosine_sim(v1, v2)) < 0.001, "Orthogonal vectors should have similarity 0.0"
print("Exercise 1 passed! ✓")

# --- Quick Check: Embeddings vs One-Hot ---
# What advantage do dense embeddings have over one-hot encoding?
# a) Embeddings use less memory
# b) Embeddings capture semantic similarity (similar words → similar vectors)
# c) Embeddings are faster to compute
# d) All of the above

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'd', "Embeddings are smaller (300 dims vs 50000), capture similarity, AND are faster for downstream computation!"
print("Exercise 2 passed! ✓")

# --- Exercise 3: Embedding Parameters ---
# If an embedding has 300 dimensions and vocabulary is 50,000 words,
# how many total parameters (numbers) does the embedding table contain?

# YOUR CODE HERE:
n_params = None  # vocab_size × embedding_dim

# --- Check ---
assert n_params is not None, "Compute the number!"
assert n_params == 15_000_000, f"Expected 50000 × 300 = 15,000,000, got {n_params:,}"
print(f"Exercise 3 passed! ✓  ({n_params:,} parameters)")

print("\n🎉 All exercises passed!")

## 🎯 Summary

**Embeddings** represent words as dense vectors:
- **Dense**: Typically 50-300 dimensions (not 50,000+)
- **Learned**: Similar words get similar vectors
- **Meaningful**: Capture semantic relationships

Key operations:
- `nn.Embedding(vocab_size, embedding_dim)` - Create embedding layer
- `embedding(token_indices)` - Look up embeddings
- Pre-trained embeddings save training time

**Next up**: Word2Vec - the algorithm that made this possible! →